In [ ]:
%load_ext autoreload
%autoreload 2

from tgraasp import STx_encoder, STx_discriminator, STx_ARGVA
from tgraasp import DatasetScheme, TGraaspDataBuilder

sample_ids = ['P0', 'P1', 'P2','P3','P4']
raw_dir    = '/home/ubuntu/sda1/qiyuan/fengyang/soft_final/T-GRAASP/T-GRAASP/data/'
builder = TGraaspDataBuilder(raw_dir=raw_dir)
graph_scheme = DatasetScheme(
    counts_tmpl="{sid}_hvg_sp_data.txt",
    labels_tmpl="{sid}_meta_sp.txt",
    positions_tmpl="{sid}_position_sp.txt",
    label_col="integrated_snn_res.0.4",
    sep=r"\s+",
    with_edges=True,
    coord_cols=(1, 2), 
)
sp_graph_list, sp_adj_list = builder.build_dataset(
    sample_ids=sample_ids,
    scheme=graph_scheme,
    test_ratio=0.1,
    distance_ref_k=100,
    distance_ref_rank=8,
    distance_margin=0.0,
)

basic_scheme = DatasetScheme(
    counts_tmpl="{sid}_hvg_data.txt",
    labels_tmpl="{sid}_meta.txt",
    sep=r"\s+",
    with_edges=False,
)
sp_basic_list = builder.build_dataset(
    sample_ids=sample_ids,
    scheme=basic_scheme,
)

N_nodes   = sp_graph_list[0].x.size(0)   # 节点个数
F_inputs  = sp_graph_list[0].x.size(1)   

ppi_file     = f'{raw_dir}/PPI.connect1.txt'
connectivity = builder.load_ppi_connectivity(ppi_file, F_inputs)    # shape = [2, E]



In [ ]:
encoder = STx_encoder(
    in_channels      = F_inputs,        
    hidden_channels  = 96,
    out_channels     = 16,
    m = 18, l = 3, K = 3,
    connect = connectivity,        
    pi = 0.75,
    n_heads = 8,
)

discriminator = STx_discriminator(
    in_channels     = 16,  
    hidden_channels = 64,   
    out_channels    = 1
)

model = STx_ARGVA(
    encoder      = encoder,
    discriminator= discriminator,
    l            = 3          
)

In [ ]:

model.load_state_dict(torch.load(
    '/home/ubuntu/sda1/qiyuan/fengyang/GBM/raw_initial/result/best_model_knn.pkl',
    map_location='cuda:1'
))
# model.load_state_dict(torch.load('/home/ubuntu/sda1/qiyuan/fengyang/GBM/raw_initial/result/best_model_knn.pkl'))


sp_graphs = sp_graph_list
sc_graphs = [sp_basic_list[0]]
sp_adjs   = sp_adj_list


trainer = TGraaspTrainer(
    model=model,
    sp_graph_list=sp_graphs,
    F_inputs=F_inputs,
    sc_graph_list=sc_graphs,
    sp_adj_list=sp_adjs,
    device="cuda:1",
    pretrain_epochs=50,
    save_dir=f'{raw_dir}/finetuned_result',
    log_dir=f'{raw_dir}/finetuned_result'
)

best_model_state = trainer.finetune(
    epochs=200,
    patience=100,
    lr=1e-5,
    lambda_recon=10.0,
    lambda_mse=0.5,
    eval_spatial_idx=0
)
